# Clip-on card tray

A first parametric CadQuery model. Units are millimetres. The tray is 420 mm wide by 300 mm deep. The floor hangs slightly below the tabletop; the perimeter rim rises a little above it.

Run cells from top to bottom. In the final model cell, rotate and zoom the 3D view. Change dimensions in section 1 as you learn.

## 1. Dimensions

The table measurements supplied so far are 24 mm top thickness and 35 mm overhang from the skirt. The 8 mm drop, clearances, skirt brace height and organizer sizes are starter values; adjust them before printing.

In [8]:
import cadquery as cq
from IPython.display import display

tray_width = 420.0
tray_depth = 300.0
floor_thickness = 2.0
tray_drop = 4.0             # floor top is this far below table top
wall_thickness = 3.0
wall_height = 12.0          # rim top is 4 mm above table top

table_thickness = 24.0
table_overhang = 35.0       # from table edge inward to skirt face
clip_clearance = 1.2
clip_thickness = 5.0
clip_width = 50.0
skirt_gap = 0.5             # set to 0 for a touching support pad
skirt_brace_height = 25.0

organizer_height = 12.0
organizer_thickness = 3.0

## 2. Tray floor and rim

`box` makes a centred rectangular solid. `translate` places it in the model. `union` joins touching or overlapping solids into one part.

In [9]:
def block(length_x, length_y, height, x, y, z):
    return (cq.Workplane("XY")
            .box(length_x, length_y, height)
            .translate((x, y, z)))

floor_top = -tray_drop
floor_bottom = floor_top - floor_thickness
floor = block(tray_width, tray_depth, floor_thickness,
              0, tray_depth / 2, (floor_top + floor_bottom) / 2)
wall_z = floor_top + wall_height / 2
side_x = tray_width / 2 - wall_thickness / 2

tray = floor
tray = tray.union(block(wall_thickness, tray_depth, wall_height,
                        -side_x, tray_depth / 2, wall_z))
tray = tray.union(block(wall_thickness, tray_depth, wall_height,
                        side_x, tray_depth / 2, wall_z))
tray = tray.union(block(tray_width, wall_thickness, wall_height,
                        0, wall_thickness / 2, wall_z))
tray = tray.union(block(tray_width, wall_thickness, wall_height,
                        0, tray_depth - wall_thickness / 2, wall_z))

display(tray)

## 3. Clips and skirt braces

Each clip has a top tab, an outer web and a lower tab. The under-table reach ends at the skirt face; a vertical pad on each clip can brace against that skirt.

In [11]:
clip_centres_x = tray_width / 2 - clip_width / 2 - 18.0
lower_top = -(table_thickness + clip_clearance)
lower_bottom = lower_top - clip_thickness
web_bottom = min(lower_bottom, floor_bottom) - 1.0
web_height = floor_top + wall_height - web_bottom
web_z = (floor_top + wall_height + web_bottom) / 2
skirt_face_y = -table_overhang
brace_y = skirt_face_y + skirt_gap + clip_thickness / 2
brace_z = -table_thickness - skirt_brace_height / 2

for x in (-clip_centres_x, clip_centres_x):
    top_tab = block(clip_width, table_overhang, floor_thickness,
                    x, -table_overhang / 2, floor_thickness / 2)
    lower_tab = block(clip_width, table_overhang, clip_thickness,
                      x, -table_overhang / 2,
                      (lower_top + lower_bottom) / 2)
    web = block(clip_width, clip_thickness, web_height,
                x, clip_thickness / 2, web_z)
    brace = block(clip_width, clip_thickness, skirt_brace_height,
                  x, brace_y, brace_z)
    tray = tray.union(top_tab).union(lower_tab).union(web).union(brace)

display(tray)

## 4. Card and accessory bays

The card bay is about 90 × 110 mm outside its walls. Three smaller bays are placeholders for tokens or dice. The rest of the tray remains open. We can resize these for your actual components.

In [12]:
def rail_x(x0, x1, y):
    return block(x1 - x0, organizer_thickness, organizer_height,
                 (x0 + x1) / 2, y, floor_top + organizer_height / 2)

def rail_y(x, y0, y1):
    return block(organizer_thickness, y1 - y0, organizer_height,
                 x, (y0 + y1) / 2, floor_top + organizer_height / 2)

walls = []
card_x0, card_x1 = -190.0, -100.0
card_y0, card_y1 = 18.0, 128.0
walls += [rail_y(card_x0, card_y0, card_y1),
          rail_y(card_x1, card_y0, card_y1),
          rail_x(card_x0, card_x1, card_y0),
          rail_x(card_x0, card_x1, card_y1)]

bay_edges = [-82.0, 0.0, 82.0, 190.0]
bay_y0, bay_y1 = 18.0, 88.0
for x0, x1 in zip(bay_edges[:-1], bay_edges[1:]):
    walls += [rail_y(x0, bay_y0, bay_y1),
              rail_y(x1, bay_y0, bay_y1),
              rail_x(x0, x1, bay_y0),
              rail_x(x0, x1, bay_y1)]

for wall in walls:
    tray = tray.union(wall)

display(tray)

## 5. Export

STEP keeps editable CAD geometry; STL is a mesh for a slicer. Export only after checking the fit and organizer sizes.

In [ ]:
cq.exporters.export(tray, "clip_on_tray.step")
cq.exporters.export(tray, "clip_on_tray.stl")